# Scrapers for MyDramaList

You will follow the instructions in Part 4 of Week 2 Tasks. In the top part of the notebook summarize through a table of content what you decided to do and then explain why.

**Author:** Kelly Chen\
**Date:** 9/15/26

**Table of Contents:**
1. [Research Question](#sec1)
2. [Scraping Aggregated Top Dramas](#sec2)
3. [Scraping Drama Page Details ](#sec3)
4. [Route A](#sec4)

<a id="sec1"></a>

## 1. Research Question

I am interested in a variety of different genres of shows, ranging from historical fantasy and romance to thriller. As many of my favorite shows are of different genres, I would like to see how this correlates to dramas on MyDramaList.

Therefore, my research question looks into what **genres** dominate or are most common in the **top dramas** ranked on MyDramaList.

To do this, I will first scrape the top dramas on MyDramaList, extrating title, rank, type (Korean, Chinese, etc.), genre, year, rating, and number of episodes. I will focus on the genre and rank to answer my research question. One more interesting thing to explore and add to my research question is how the genres for **each type** of drama on the top-ranked list vary.

<a id="sec2"></a>

## 2. Scraping Aggregated Top Dramas 

In [1]:
import requests
from bs4 import BeautifulSoup
from seleniumbase import Driver
import math
import re
import json

#### Try Using Selenium first

Add the parse_drama() function used previously here and modify as needed for scraping from MyDramaList:

In [2]:
def sel_parse_drama(card):
    """Parses HTML content and returns information on drama card on each page"""
    soup = BeautifulSoup(card.get_attribute("outerHTML"), "html.parser")

    ranking = soup.find("div", class_="ranking").get_text(strip=True)
    title = soup.find("h6", class_="title").get_text(strip=True)

    meta = soup.find("span", class_="text-muted").get_text(strip=True)
    type, year, eps = re.findall(r"(.+?) - (\d{4}), (.+)", meta)[0] #use regex

    rating = soup.find("span", class_="score").get_text(strip=True)

    return {
        "rank": ranking,
        "title": title,
        "year": year,
        "eps": eps,
        "rating": rating,
        "type": type
    }

In [159]:
url = "https://mydramalist.com/shows/top"

drama_data = []

with Driver(browser="firefox") as driver:
    driver.open(url)

    # 1. Extract total results count 
    total_text = driver.get_text("p.m-b-sm.pull-right") # This will be a string value
    total_results = int(total_text.split()[0])
    #print(total_results) #5000 results

    # 2. Calculate total pages (20 items per page)
    total_pages = math.ceil(total_results / 20)
    print(f"Total results: {total_results} | Total pages: {total_pages}")

    # 4. Loop through each page URL
    for page in range(1, total_pages + 1):
        driver.open(f"{url}?page={page}")
        driver.sleep(1.0)

        # 5. Extract items on the current page
        #cards = driver.find_elements(".box") # found 23 but only 20 dramas; need to get drama cards only

        #ensures .box elements contains dramas only by getting container around dramas  
        cards = driver.find_elements("div.m-t.nav-active-border.b-primary .box")

        print(f"Page {page}: Scraped {len(cards)} dramas")

        # 6. extract info with sel_parse_drama()
        for card in cards:
            drama = sel_parse_drama(card)
            drama_data.append(drama)

with open("top_dramas_selenium.json", "w") as f:
   json.dump(drama_data, f, indent=4)

Total results: 5000 | Total pages: 250
Page 1: Scraped 20 dramas
Page 2: Scraped 20 dramas
Page 3: Scraped 20 dramas
Page 4: Scraped 20 dramas
Page 5: Scraped 20 dramas
Page 6: Scraped 20 dramas
Page 7: Scraped 20 dramas
Page 8: Scraped 20 dramas
Page 9: Scraped 20 dramas
Page 10: Scraped 20 dramas
Page 11: Scraped 20 dramas
Page 12: Scraped 20 dramas
Page 13: Scraped 20 dramas
Page 14: Scraped 20 dramas
Page 15: Scraped 20 dramas
Page 16: Scraped 20 dramas
Page 17: Scraped 20 dramas
Page 18: Scraped 20 dramas
Page 19: Scraped 20 dramas
Page 20: Scraped 20 dramas
Page 21: Scraped 20 dramas
Page 22: Scraped 20 dramas
Page 23: Scraped 20 dramas
Page 24: Scraped 20 dramas
Page 25: Scraped 20 dramas
Page 26: Scraped 20 dramas
Page 27: Scraped 20 dramas
Page 28: Scraped 20 dramas
Page 29: Scraped 20 dramas
Page 30: Scraped 20 dramas
Page 31: Scraped 20 dramas
Page 32: Scraped 20 dramas
Page 33: Scraped 20 dramas
Page 34: Scraped 20 dramas
Page 35: Scraped 20 dramas
Page 36: Scraped 20 drama

**Fixed** 
At first, selenium could open the website but it seemed to time out; selenium can open the url but cannot parse the content because it could not find the object I was trying to scrape. Therefore tried scraping with BeautifulSoup instead.

#### Try BeautifulSoup to get HTML content:

In [3]:
def fetch_page_content(url):
    """Fetches HTML content from a URL and checks status code."""
    response = requests.get(url)

    print(f"URL: {url}")
    print(f"Status Code: {response.status_code}")

    if response.status_code == 200:
        return response.text
    else:
        print(f"Failed to fetch page. Status code: {response.status_code}")
        return None

In [4]:
def parse_drama(html_content):
    """Parses HTML content and returns information on dramas as a list of dictionaries"""
    if not html_content:
        return []

    soup = BeautifulSoup(html_content, "html.parser") 
    
    container = soup.find("div", class_="m-t nav-active-border b-primary")
    cards = container.find_all("div", class_="box")
    #print(len(cards)) 20 cards on each page; 5000 total

    drama_data = []

    for card in cards:
        title = card.find("h6", class_="title").get_text(strip=True)
      
        ranking = card.find("div", class_="ranking").get_text(strip=True)
        
        meta = card.find("span", class_="text-muted").get_text(strip=True) # includes type, year, and # eps
        #type, year, eps = re.findall(r"(^\w+? Drama) - (\d{4}), (\d+)", meta) # tried use regex 
        
        rating = card.find("span", class_="score").get_text(strip=True)
        
        drama = {
            "title": title,
            "rank": ranking,
            "meta": meta,
            "rating": rating
        }
        drama_data.append(drama)
    return drama_data
   

In [175]:
#page = fetch_page_content(url)
#print(page)

all_dramas = []

for page_num in range(1, 251):
    url = f"https://mydramalist.com/shows/top?page={page_num}"
    
    page = fetch_page_content(url)
    dramas = parse_drama(page)
    
    all_dramas.extend(dramas) #returns one list instead of lists of lists with append

print(len(all_dramas))

URL: https://mydramalist.com/shows/top?page=1
Status Code: 200
URL: https://mydramalist.com/shows/top?page=2
Status Code: 200
URL: https://mydramalist.com/shows/top?page=3
Status Code: 200
URL: https://mydramalist.com/shows/top?page=4
Status Code: 200
URL: https://mydramalist.com/shows/top?page=5
Status Code: 200
URL: https://mydramalist.com/shows/top?page=6
Status Code: 200
URL: https://mydramalist.com/shows/top?page=7
Status Code: 200
URL: https://mydramalist.com/shows/top?page=8
Status Code: 200
URL: https://mydramalist.com/shows/top?page=9
Status Code: 200
URL: https://mydramalist.com/shows/top?page=10
Status Code: 200
URL: https://mydramalist.com/shows/top?page=11
Status Code: 200
URL: https://mydramalist.com/shows/top?page=12
Status Code: 200
URL: https://mydramalist.com/shows/top?page=13
Status Code: 200
URL: https://mydramalist.com/shows/top?page=14
Status Code: 200
URL: https://mydramalist.com/shows/top?page=15
Status Code: 200
URL: https://mydramalist.com/shows/top?page=16
St

Save into a csv file:

In [176]:
import csv

with open("top_dramas_bs.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=all_dramas[0].keys())
    writer.writeheader()
    writer.writerows(all_dramas)

<a id="sec3"></a>

## 3. Scraping Drama Page Details

Write a function for scraping drama details:

In [5]:
def parse_drama_details(html_content):
    """Parses HTML content and returns drama details as a dictionary"""
    if not html_content:
        return {}

    soup = BeautifulSoup(html_content, "html.parser") 
    
    details = soup.find("div", class_="col-sm-8")
    #print(details)
    
    rating = details.find("div", class_="col-film-rating", id="show-detailsxx").get_text(strip=True) #number rating only

    # aggregate rating
    hfs = details.find_all("div", class_="hfs")
    ratings = hfs[0].get_text(" ", strip=True) # index to get each element and add space between separate elements in text
    watchers = hfs[1].get_text(" ", strip=True)
    reviews = hfs[2].get_text(" ", strip=True)

    synopsis = details.find("div", class_="show-synopsis").find("p").get_text(strip=True) #within <p></p> tag

    #lists
    lists = details.find_all("li", class_="list-item p-a-0")
    #print(lists) check list to see if correct
    native_title = lists[0].get_text(" ", strip=True)
    other_names = lists[1].get_text(" ", strip=True)
    screenwriter = lists[2].get_text(" ", strip=True)
    director = lists[3].get_text(" ", strip=True)

    genres = details.find("li", class_="list-item p-a-0 show-genres").get_text(" ", strip=True)
    
    tags = details.find("li", class_="list-item p-a-0 show-tags").get_text(" ", strip=True)

    drama_details = {
        "rating": rating,
        "aggregate_ratings": ratings,
        "watchers": watchers,
        "reviews": reviews,
        "synopsis": synopsis,
        "native_title": native_title,
        "other_names": other_names,
        "screenwriter": screenwriter,
        "director": director,
        "genres": genres,
        "tags": tags
    }

    return drama_details


Parsing drama details for *Pursuit of Jade*:

In [ ]:
url = "https://mydramalist.com/760409-zhu-yu" #page for Pursuit of Jade

page = fetch_page_content(url)
details = parse_drama_details(page)
#print(details)
for key, value in details.items():
    print(f"{key}: {value}")

URL: https://mydramalist.com/760409-zhu-yu
Status Code: 200
rating: 9.1
aggregate_ratings: Ratings: 9.1 /10 from 42,737 users
watchers: # of Watchers: 82,629
reviews: Reviews: 656 users
synopsis: It follows Fan Chang Yu, a butcher’s daughter, and Xie Zheng, a fallen noble seeking revenge. Their fake marriage turns into true love, but war tears them apart. Determined, Fan Chang Yu wields her butcher’s knife on the battlefield, searching for justice and her husband. Meanwhile, Xie Zheng reclaims his title, fighting to protect his country and love. Reunited in battle, they stand together, defying fate and uncovering the truth.

(Source: WeTV)~~ Adapted from the web novel "Zhu Yu" (逐玉) by Tuan Zi Lai Xi (团子来袭).Edit Translation
native_title: Native Title: 逐玉
other_names: Also Known As: Chasing Jade ,  Zhu Yu
screenwriter: Screenwriter: Zou Yue
director: Director: Zeng Qing Jie
genres: Genres: Historical , Mystery , Romance , War
tags: Tags: Fake To Real Lovers , Physically Strong Female Lea

In [164]:
# save drama details
with open("poj_drama_details.json", "w", encoding="utf-8") as f:
    json.dump(details, f, indent=4)

#### Get other recommendations based on the drama

In [6]:
def parse_recommendations(html_content):
    """Parses HTML content and returns recommendations from a drama page"""
    if not html_content:
        return []
    
    soup = BeautifulSoup(html_content, "html.parser") 

    recommendations = [] 
    rec_container = soup.find("div", class_="details-recommendations")

    items = rec_container.find_all("div", class_="rec-item")
   
    for item in items:
        link = item.find("a")
        img = item.find("img")

        #title = link.get("oldtitle")
        title = img.get("alt")
        url = link.get("href")

        recommendations.append(
            {
            "title": title,
            "url": f"https://mydramalist.com{url}"
            }
        )

    return recommendations

In [7]:
url = "https://mydramalist.com/760409-zhu-yu" #page for Pursuit of Jade

page = fetch_page_content(url)
recs = parse_recommendations(page)
print(recs)

URL: https://mydramalist.com/760409-zhu-yu
Status Code: 200
[{'title': 'Legend of the Female General', 'url': 'https://mydramalist.com/754387-legend-of-the-female-general'}, {'title': 'Are You the One', 'url': 'https://mydramalist.com/699401-jiao-cang'}, {'title': 'Blossom', 'url': 'https://mydramalist.com/745395-jiu-chong-zi'}, {'title': 'Fated Hearts', 'url': 'https://mydramalist.com/768987-wan-xin-ji'}, {'title': 'The Prisoner of Beauty', 'url': 'https://mydramalist.com/687393-the-prisoner-of-beauty'}, {'title': 'Glory', 'url': 'https://mydramalist.com/772283-ming-men-shi-jia'}]


Save into JSON file:

In [172]:
with open("poj_other_drama_recs.json", "w", encoding="utf-8") as f:
    json.dump(recs, f, indent=4)

<a id="sec4"></a>

## 4. Route A
#### Compare Top Ranked vs. Most Popular Dramas and their Corresponding Top Genres

I already have the list of the 5000 top ranked dramas (saved in csv file). However, I do not have the genres for them, and I also need to get the names and genres for the most popular dramas. Therefore, I need to get URLs for each drama and click into them to get the genre details.

Use selenium to scrape list of most popular dramas:

In [8]:
url = "https://mydramalist.com/shows/popular"

pop_dramas = []

with Driver(browser="firefox") as driver:
    driver.open(url)

    # 1. Extract total results count 
    total_text = driver.get_text("p.m-b-sm.pull-right") # This will be a string value
    total_results = int(total_text.split()[0])

    # 2. Calculate total pages (20 items per page)
    total_pages = math.ceil(total_results / 20)
    print(f"Total results: {total_results} | Total pages: {total_pages}")

    # 4. Loop through each page URL
    for page in range(1, total_pages + 1):
        driver.open(f"{url}?page={page}")
        driver.sleep(0.2)

        # get HTML from selenium
        soup = BeautifulSoup(driver.page_source, "html.parser")

        # 5. Extract items on the current page
 
        container = soup.find("div", class_="m-t nav-active-border b-primary")
        cards = container.find_all("div", class_="box")

        # 6. extract info with sel_parse_drama()
        for card in cards:
            title = card.find("h6", class_="title").get_text(strip=True)

            link = card.find("a", href=True)
            drama_url = f"https://mydramalist.com{link['href']}"

            pop_dramas.append({
                "title": title,
                "url": drama_url
            })

        print(f"Page {page}: Scraped {len(cards)} dramas")
        
with open("popular_dramas_selenium.json", "w") as f:
   json.dump(pop_dramas, f, indent=4)

Total results: 5000 | Total pages: 250
Page 1: Scraped 20 dramas
Page 2: Scraped 20 dramas
Page 3: Scraped 20 dramas
Page 4: Scraped 20 dramas
Page 5: Scraped 20 dramas
Page 6: Scraped 20 dramas
Page 7: Scraped 20 dramas
Page 8: Scraped 20 dramas
Page 9: Scraped 20 dramas
Page 10: Scraped 20 dramas
Page 11: Scraped 20 dramas
Page 12: Scraped 20 dramas
Page 13: Scraped 20 dramas
Page 14: Scraped 20 dramas
Page 15: Scraped 20 dramas
Page 16: Scraped 20 dramas
Page 17: Scraped 20 dramas
Page 18: Scraped 20 dramas
Page 19: Scraped 20 dramas
Page 20: Scraped 20 dramas
Page 21: Scraped 20 dramas
Page 22: Scraped 20 dramas
Page 23: Scraped 20 dramas
Page 24: Scraped 20 dramas
Page 25: Scraped 20 dramas
Page 26: Scraped 20 dramas
Page 27: Scraped 20 dramas
Page 28: Scraped 20 dramas
Page 29: Scraped 20 dramas
Page 30: Scraped 20 dramas
Page 31: Scraped 20 dramas
Page 32: Scraped 20 dramas
Page 33: Scraped 20 dramas
Page 34: Scraped 20 dramas
Page 35: Scraped 20 dramas
Page 36: Scraped 20 drama

Write function to parse each drama from url and get genres:

In [9]:
def parse_genres(html_content):
    """Parses HTML content for each drama and returns genres"""
    if not html_content:
        return []
    
    soup = BeautifulSoup(html_content, "html.parser")
    details = soup.find("div", class_="show-detailsxss")

    genres = details.find("li", class_="list-item p-a-0 show-genres").get_text(" ", strip=True)
  
    return genres

Now, use selenium to visit each url and get genres:

In [11]:
with Driver(browser="firefox") as driver:

    for i, drama in enumerate(pop_dramas):
        driver.open(drama["url"]) #open url from the list
        driver.sleep(0.1)

        drama["genres"] = parse_genres(driver.page_source) # gets genres by calling function and adds to list

        print(f"{i + 1}/{len(pop_dramas)}: {drama['title']}") # prints progress

1/5000: Guardian: The Lonely and Great God
2/5000: Strong Girl Bong Soon
3/5000: It's Okay to Not Be Okay
4/5000: Crash Landing on You
5/5000: What's Wrong with Secretary Kim
6/5000: Business Proposal
7/5000: Descendants of the Sun
8/5000: Weightlifting Fairy Kim Bok Joo
9/5000: W
10/5000: True Beauty
11/5000: Vincenzo
12/5000: Weak Hero Class 1
13/5000: Alchemy of Souls
14/5000: Hotel del Luna
15/5000: Twinkling Watermelon
16/5000: Flower of Evil
17/5000: Squid Game
18/5000: While You Were Sleeping
19/5000: Boys over Flowers
20/5000: The Heirs
21/5000: Moon Lovers: Scarlet Heart Ryeo
22/5000: My Love from the Star
23/5000: Healer
24/5000: Lovely Runner
25/5000: Extraordinary Attorney Woo
26/5000: Hometown Cha-Cha-Cha
27/5000: My ID Is Gangnam Beauty
28/5000: Itaewon Class
29/5000: The Legend of the Blue Sea
30/5000: Extraordinary You
31/5000: I'm Not a Robot
32/5000: The Glory
33/5000: Fight for My Way
34/5000: Move to Heaven
35/5000: Pinocchio
36/5000: My Demon
37/5000: Reply 1988
38

AttributeError: 'NoneType' object has no attribute 'get_text'

Save updated pop_drams JSON file:

In [12]:
with open("popular_dramas_selenium.json", "w") as f:
   json.dump(pop_dramas, f, indent=4)